# Inference
This notebook loads the trained model and runs inference on test DICOM files.

In [1]:
import numpy as np
import pydicom
from skimage.transform import resize
from keras.models import model_from_json
from keras.applications.vgg16 import VGG16
from keras.models import Sequential
from keras.layers import Dense, Dropout, Flatten
import os

Using TensorFlow backend.


In [3]:
def check_dicom(filename):
    print(f'Load file {filename} ...')
    try:
        ds = pydicom.dcmread(filename)
    except Exception as e:
        print(f'  Error reading DICOM: {e}')
        return None

    modality = getattr(ds, 'Modality', None)
    if modality != 'DX':
        print(f'  Rejected: Modality is {modality}, expected DX')
        return None

    body_part = getattr(ds, 'BodyPartExamined', None)
    if body_part and body_part.upper() not in ['CHEST', 'THORAX']:
        print(f'  Rejected: BodyPartExamined is {body_part}, expected CHEST')
        return None

    view_pos = getattr(ds, 'ViewPosition', None)
    patient_pos = getattr(ds, 'PatientPosition', None)
    valid_positions = ['AP', 'PA']
    
    if view_pos is None and patient_pos is None:
        print(f'  Warning: ViewPosition and PatientPosition both missing')
    elif view_pos and view_pos.upper() not in valid_positions:
        if not patient_pos or patient_pos.upper() not in valid_positions:
            print(f'  Rejected: ViewPosition={view_pos}, PatientPosition={patient_pos}')
            return None

    print(f'  Accepted: Modality={modality}, BodyPart={body_part}, View={view_pos}')
    return ds.pixel_array


def preprocess_image(img, img_size):
    if len(img.shape) == 2:
        img = np.stack([img] * 3, axis=-1)
    img_resized = resize(img, (img_size[1], img_size[2]), anti_aliasing=True)
    img_rescaled = img_resized.astype(np.float32)
    return np.expand_dims(img_rescaled, axis=0)


def load_model(model_path, weight_path):
    base_model = VGG16(include_top=False, weights=None, input_shape=(224, 224, 3))
    for layer in base_model.layers[:-4]:
        layer.trainable = False
    for layer in base_model.layers[-4:]:
        layer.trainable = True

    model = Sequential([
        base_model,
        Flatten(),
        Dense(1024, activation='relu'),
        Dropout(0.5),
        Dense(512, activation='relu'),
        Dropout(0.5),
        Dense(256, activation='relu'),
        Dropout(0.3),
        Dense(1, activation='sigmoid')
    ])
    model.load_weights(weight_path)
    print(f'Loaded weights from {weight_path}')
    return model


def predict_image(model, img, thresh):
    pred_prob = model.predict(img, verbose=0)[0][0]
    prediction = 'Pneumonia' if pred_prob >= thresh else 'No Pneumonia'
    print(f'  Probability: {pred_prob:.3f} -> {prediction}')
    return prediction

In [4]:
IMG_SIZE = (1, 224, 224, 3)
thresh = 0.523
weight_path = 'my_model_weights.h5'

my_model = load_model(None, weight_path)

test_dicoms = ['test1.dcm', 'test2.dcm', 'test3.dcm', 'test4.dcm', 'test5.dcm', 'test6.dcm']

for i in test_dicoms:
    img = check_dicom(i)
    if img is None:
        print(f'  Skipping {i}\n')
        continue
    img_proc = preprocess_image(img, IMG_SIZE)
    pred = predict_image(my_model, img_proc, thresh)
    print(f'Result for {i}: {pred}\n')

Loaded weights from my_model_weights.h5
Load file test1.dcm ...
  Accepted: Modality=DX, BodyPart=CHEST, View=None
  Probability: 0.269 -> No Pneumonia
Result for test1.dcm: No Pneumonia

Load file test2.dcm ...
  Accepted: Modality=DX, BodyPart=CHEST, View=None
  Probability: 0.748 -> Pneumonia
Result for test2.dcm: Pneumonia

Load file test3.dcm ...
  Accepted: Modality=DX, BodyPart=CHEST, View=None
  Probability: 0.838 -> Pneumonia
Result for test3.dcm: Pneumonia

Load file test4.dcm ...
  Rejected: BodyPartExamined is RIBCAGE, expected CHEST
  Skipping test4.dcm

Load file test5.dcm ...
  Rejected: Modality is CT, expected DX
  Skipping test5.dcm

Load file test6.dcm ...
  Accepted: Modality=DX, BodyPart=CHEST, View=None
  Probability: 0.269 -> No Pneumonia
Result for test6.dcm: No Pneumonia

